# FAIR²-FoodHealth Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and explore a Croissant-described dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL below:

- `https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json`

### Goal
We will:
1. Load the dataset metadata.
2. Inspect record sets and fields using their `@id` values.
3. Load selected record sets into pandas DataFrames.
4. Perform exploratory data analysis (EDA) using dynamically selected fields.
5. Create visualizations for numeric and categorical summaries.

> Important: all dataset entities in this notebook are referenced by their `@id` values whenever available.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and initialize the dataset object from the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully.")
print("Name:", getattr(metadata, "name", "N/A"))
print("Description:", getattr(metadata, "description", "N/A"))
print("Version:", getattr(metadata, "version", "N/A"))
print("License:", getattr(metadata, "license", "N/A"))
print("Citation:", getattr(metadata, "cite_as", getattr(metadata, "citeAs", "N/A")))

## 2. Metadata Overview
Inspect high-level metadata safely. In this notebook, `dataset.metadata` is treated as a single object rather than as a dictionary or list.

In [ ]:
# Show selected metadata attributes if available
metadata_attributes = [
    "name",
    "description",
    "version",
    "url",
    "license",
    "date_published",
    "keywords",
]

for attr in metadata_attributes:
    print(f"{attr}: {getattr(metadata, attr, 'N/A')}")

## 3. Record Set Overview
Review the available record sets and their field definitions. We extract `@id` values dynamically from the metadata JSON representation for exploration.

In [ ]:
# Convert metadata to JSON for structured inspection of record sets and fields
metadata_json = metadata.to_json()
record_sets = metadata_json.get("recordSet", [])

print(f"Number of record sets discovered: {len(record_sets)}")

record_set_summaries = []
for rs in record_sets:
    rs_id = rs.get("@id", "N/A")
    rs_name = rs.get("name", rs_id)
    rs_desc = rs.get("description", "")
    field_count = len(rs.get("field", []))
    record_set_summaries.append({
        "record_set_id": rs_id,
        "record_set_name": rs_name,
        "field_count": field_count,
        "description": rs_desc,
    })

record_sets_df = pd.DataFrame(record_set_summaries)
record_sets_df

If record sets are available, inspect the fields of the first record set by `@id`.

In [ ]:
if len(record_sets) == 0:
    print("No record sets were listed in the metadata.")
else:
    first_record_set = record_sets[0]
    first_record_set_id = first_record_set.get("@id")
    print("First record set @id:", first_record_set_id)
    
    field_rows = []
    for field in first_record_set.get("field", []):
        field_rows.append({
            "field_id": field.get("@id", "N/A"),
            "field_name": field.get("name", ""),
            "data_type": field.get("dataType", ""),
            "description": field.get("description", ""),
            "source": field.get("source", ""),
        })
    
    fields_df = pd.DataFrame(field_rows)
    display(fields_df)

## 4. Sampling Records by Record Set `@id`
Use a record set `@id` to access records. This mirrors the core `mlcroissant` workflow while keeping identifiers explicit.

In [ ]:
if len(record_sets) == 0:
    print("No record sets available for sampling.")
else:
    sample_record_set_id = record_sets[0].get("@id")
    print(f"Sampling records from record set: {sample_record_set_id}")
    
    try:
        for i, record in enumerate(dataset.records(record_set=sample_record_set_id)):
            print(record)
            if i >= 2:
                break
    except Exception as e:
        print("Could not sample records from this record set.")
        print(type(e).__name__, str(e))

## 5. Data Extraction
Load records from selected record sets into pandas DataFrames. The notebook uses variables so the workflow stays reusable even if record set IDs change across datasets.

In [ ]:
# Select a small number of record sets for extraction
# We use discovered @id values directly.
record_set_ids = [rs.get("@id") for rs in record_sets if rs.get("@id")]
selected_record_set_ids = record_set_ids[:3]

print("Selected record set @id values:")
for rs_id in selected_record_set_ids:
    print("-", rs_id)

dataframes = {}
load_status = []

for rs_id in selected_record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        load_status.append({
            "record_set_id": rs_id,
            "rows": len(df),
            "columns": len(df.columns),
            "status": "loaded"
        })
    except Exception as e:
        load_status.append({
            "record_set_id": rs_id,
            "rows": 0,
            "columns": 0,
            "status": f"failed: {type(e).__name__}: {e}"
        })

load_status_df = pd.DataFrame(load_status)
load_status_df

In [ ]:
# Inspect the first successfully loaded DataFrame
loaded_record_sets = [rs_id for rs_id, df in dataframes.items() if not df.empty or len(df.columns) > 0]

if not loaded_record_sets:
    print("No record sets were successfully loaded into DataFrames.")
else:
    active_record_set_id = loaded_record_sets[0]
    active_df = dataframes[active_record_set_id]
    
    print("Active record set @id:", active_record_set_id)
    print("Columns:")
    print(active_df.columns.tolist())
    display(active_df.head())

## 6. Field Inventory for Loaded Record Sets
Create a lookup table of field `@id` values to help connect metadata definitions with loaded DataFrame columns.

In [ ]:
field_inventory = []

for rs in record_sets:
    rs_id = rs.get("@id", "N/A")
    for field in rs.get("field", []):
        field_inventory.append({
            "record_set_id": rs_id,
            "field_id": field.get("@id", "N/A"),
            "field_name": field.get("name", ""),
            "data_type": field.get("dataType", ""),
            "description": field.get("description", ""),
        })

field_inventory_df = pd.DataFrame(field_inventory)
field_inventory_df.head(20)

## 7. Exploratory Data Analysis (EDA)
Apply common data processing steps on one loaded record set:
- detect numeric columns,
- filter records using a threshold,
- normalize a numeric field,
- optionally group by a categorical field.

The analysis is written defensively so it can adapt to whichever record sets are actually exposed by the dataset.

In [ ]:
def find_numeric_column(df):
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    if numeric_cols:
        return numeric_cols[0]
    
    # Attempt conversion for object columns
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().sum() > 0:
            df[col] = converted
            return col
    return None

def find_group_column(df, exclude_cols=None):
    exclude_cols = exclude_cols or []
    candidate_cols = [c for c in df.columns if c not in exclude_cols]
    object_cols = df[candidate_cols].select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    if object_cols:
        return object_cols[0]
    return None

if not loaded_record_sets:
    print("EDA skipped because no DataFrame was loaded.")
else:
    record_set_id = loaded_record_sets[0]
    eda_df = dataframes[record_set_id].copy()
    
    numeric_field_id = find_numeric_column(eda_df)
    
    if numeric_field_id is None:
        print(f"No numeric field detected in record set {record_set_id}.")
    else:
        print("Using record set @id:", record_set_id)
        print("Numeric field selected:", numeric_field_id)
        
        numeric_series = pd.to_numeric(eda_df[numeric_field_id], errors="coerce")
        valid_numeric = numeric_series.dropna()
        
        if len(valid_numeric) == 0:
            print("Selected numeric field contains no valid numeric values.")
        else:
            threshold = valid_numeric.quantile(0.5)
            filtered_df = eda_df.loc[numeric_series > threshold].copy()
            
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())
            
            filtered_numeric = pd.to_numeric(filtered_df[numeric_field_id], errors="coerce")
            std_value = filtered_numeric.std()
            
            if pd.notna(std_value) and std_value != 0:
                normalized_col = f"{numeric_field_id}_normalized"
                filtered_df[normalized_col] = (filtered_numeric - filtered_numeric.mean()) / std_value
                print(f"Normalized field: {normalized_col}")
                display(filtered_df[[numeric_field_id, normalized_col]].head())
            else:
                print("Normalization skipped because the standard deviation is zero or undefined.")
            
            group_field_id = find_group_column(filtered_df, exclude_cols=[numeric_field_id])
            if group_field_id is not None:
                print("Grouping field selected:", group_field_id)
                grouped_df = (
                    filtered_df.groupby(group_field_id)[numeric_field_id]
                    .mean()
                    .reset_index()
                    .sort_values(by=numeric_field_id, ascending=False)
                )
                display(grouped_df.head())
            else:
                print("No suitable grouping field was found.")

## 8. Missing Values and Basic Profiling
Check missingness and summarize the active DataFrame.

In [ ]:
if not loaded_record_sets:
    print("Profiling skipped because no DataFrame was loaded.")
else:
    profile_df = dataframes[loaded_record_sets[0]].copy()
    
    missing_summary = pd.DataFrame({
        "column": profile_df.columns,
        "missing_count": profile_df.isna().sum().values,
        "missing_fraction": profile_df.isna().mean().values,
        "dtype": profile_df.dtypes.astype(str).values,
    }).sort_values(by="missing_fraction", ascending=False)
    
    display(missing_summary.head(20))
    display(profile_df.describe(include="all").transpose().head(20))

## 9. Visualization
Visualize numeric distributions and, when possible, category frequencies for the selected record set.

In [ ]:
if not loaded_record_sets:
    print("Visualization skipped because no DataFrame was loaded.")
else:
    viz_df = dataframes[loaded_record_sets[0]].copy()
    numeric_field_id = find_numeric_column(viz_df)
    category_field_id = find_group_column(viz_df, exclude_cols=[numeric_field_id] if numeric_field_id else [])
    
    if numeric_field_id is not None:
        numeric_values = pd.to_numeric(viz_df[numeric_field_id], errors="coerce").dropna()
        if len(numeric_values) > 0:
            plt.figure(figsize=(8, 4))
            sns.histplot(numeric_values, bins=30, kde=True)
            plt.title(f"Distribution of {numeric_field_id}")
            plt.xlabel(numeric_field_id)
            plt.ylabel("Count")
            plt.show()
        else:
            print(f"No valid numeric values available for {numeric_field_id}.")
    else:
        print("No numeric field available for histogram.")
    
    if category_field_id is not None:
        top_categories = viz_df[category_field_id].astype(str).value_counts().head(10)
        if len(top_categories) > 0:
            plt.figure(figsize=(10, 4))
            sns.barplot(x=top_categories.index, y=top_categories.values)
            plt.title(f"Top categories in {category_field_id}")
            plt.xlabel(category_field_id)
            plt.ylabel("Frequency")
            plt.xticks(rotation=45, ha="right")
            plt.show()
        else:
            print(f"No category values available for {category_field_id}.")
    else:
        print("No categorical field available for bar chart.")

## 10. Cross-Record-Set Summary
Summarize all loaded record sets to compare shape and readiness for downstream work.

In [ ]:
if not dataframes:
    print("No DataFrames available for summary.")
else:
    summary_rows = []
    for rs_id, df in dataframes.items():
        summary_rows.append({
            "record_set_id": rs_id,
            "n_rows": len(df),
            "n_columns": len(df.columns),
            "numeric_columns": len(df.select_dtypes(include="number").columns),
            "object_columns": len(df.select_dtypes(include=["object", "category"]).columns),
        })
    
    summary_df = pd.DataFrame(summary_rows).sort_values(by=["n_rows", "n_columns"], ascending=False)
    display(summary_df)

## 11. Conclusion
In this notebook, we loaded the FAIR²-FoodHealth dataset via its Croissant schema and explored it with `mlcroissant`.

Key steps included:
- initializing the dataset from the schema URL,
- reviewing metadata without treating `dataset.metadata` as a dictionary,
- discovering record sets and fields using their `@id` values,
- loading selected record sets into pandas,
- performing adaptable EDA on available numeric and categorical fields,
- and producing simple visual summaries.

Because Croissant datasets can expose many record sets with different schemas, the notebook uses variables and defensive checks so it can adapt to the actual structure returned by the dataset.